# Анализ новостей за определенный период и его источников

1. Провел анализ по поиску адекватного API по предоставлению сводки новостей, по которым можно найти признак изменения состояния рынка в определенную дату. Пользовался следующими API: NewsAPI, Alpha Vantage, Alpha Vantage, RSS-ленты, Benzinga API.
В ходе работы столкнулся с такими проблемами, как ограниченное количество выводимых статей и отсутствие адекватной документации сервиса (в сервисе Benzinga API документация 2023 года, что вызывает конфликт между библиотек). RSS-ленты требую дополнительного парсинга, но и это решение имеет ограниченное количество выводов. Примеры выводов новостных статей приведены ниже.
2. Несмотря на вышеуказанные особенности сталкиваемся с вопросом об определении признака новости (положительно или отрицательно влияет эта новость на тот или иной раздел рынка). Сервисы не выдают данный признак, следовательно, необходимо либо разрабатывать модель распознавания тематики новости, либо определять по ключевым словам. Последнее не даст объективной оценки новости, т.к. ключевое слово "снижение" может использоваться в нескольких смыслах ("снижение ключевой ставки по ипотеке вызвало рост акций строительных компаний", "снижение уровня дохода населения повлияло на социально - экономическую устойчивость страны".
3. Можно использовать готовые датасеты с новостями из Kaggle, но там также ограниченное количество данных и отсутствует признак статьи.

По результатам поиска источников новостей можно выделить следующие выводы:
- использование сервисов API облегчает поиск информации;
- информация будет поставляться ограниченным количеством и не факт, что будет за необходимый период;
- определение признака статьи вызовет дополнительные трудозатраты, либо будет не объективным;
- использование нескольких сервисов API сразу может решить указанные проблемы, что отрицательно скажется на чистоту новостей.
- целесообразно будет использовать готовые датасеты с наличием признака новости, полученные от нейросети.

In [4]:
# вывод новостей через сервис newsapi.org
import requests
import pandas as pd
from datetime import datetime

pd.set_option('display.max_columns', None)  # Показывать все колонки
pd.set_option('display.width', 1000)        # Ширина вывода
pd.set_option('display.max_colwidth', 50)

def get_news(query, api_key):
    url = "https://newsapi.org/v2/everything"
    params = {
        'q': query,
        'language': 'en',
        'sortBy': 'publishedAt',
        'apiKey': api_key,
        'pageSize': 10
    }
    response = requests.get(url, params=params)
    return response.json()

data = get_news("finance", '5515f0d78f8f48cdb43b8b17ace93dfd')
articles_array = data['articles']
print(f"Всего статей: {len(articles_array)}")


def get_dataset(articles_array):
    df = []

    for article in articles_array:
        published_date = datetime.strptime(article['publishedAt'], '%Y-%m-%dT%H:%M:%SZ')
        df.append({
            'date': published_date,
            'title': article['title'],
            'source': article['source']['name'],
            'description': article['description'],
            'url': article['url']
        })
    return pd.DataFrame(df)

print(get_dataset(articles_array).head(10))

Всего статей: 10
                 date                                              title               source                                        description                                                url
0 2025-11-01 13:05:00  4 Steps for a Fall Finance Audit, According to...  Yahoo Entertainment  GOBankingRates talked with stock market expert...  https://finance.yahoo.com/news/4-steps-fall-fi...
1 2025-11-01 13:00:10  Interactive ad: Itaú: Itaú: Real Monsters - Ghost      Bestadsontv.com  On Halloween, monsters come out.\nSome for fun...  https://bestadsontv.com/ad/186171/Ita-Ita-Real...
2 2025-11-01 13:00:10  Top 3 cryptos to buy for 2026: Bitcoin (BTC), ...        Ambcrypto.com  Analysts have spotlighted Bitcoin (BTC), Dogec...  https://ambcrypto.com/top-3-cryptos-to-buy-for...
3 2025-11-01 13:00:00  MiCA Won’t Save Us from a Stablecoin Crisis. I...             CoinDesk  MiCA deserves credit for imposing order on cha...  https://www.coindesk.com/opinion/2025/10/31/mi...
4 2

In [5]:
# вывод новостей через RSS
import feedparser

def parse_rss_feed(feed_url):
    feed = feedparser.parse(feed_url)
    articles = []
    for entry in feed.entries:
        article = {
            'title': entry.title if hasattr(entry, 'title') else 'No title',
            'published': entry.published if hasattr(entry, 'published') else 'No date',
            'link': entry.link if hasattr(entry, 'link') else 'No link'
        }
        if hasattr(entry, 'summary'):
            article['summary'] = entry.summary
        elif hasattr(entry, 'description'):
            article['summary'] = entry.description
        else:
            article['summary'] = 'No summary'
        articles.append(article)
    return articles

feeds = {
    'Reuters Business': 'http://feeds.reuters.com/reuters/businessNews',
    'Yahoo Finance': 'https://finance.yahoo.com/news/rssindex',
}

print("Yahoo Finance:")
articles = parse_rss_feed(feeds['Yahoo Finance'])

# указываю вывести 1000 новостей, но выведет всё равно 49
for i, article in enumerate(articles[:1000], 1):
    print(f"\nСтатья {i}")
    print(f"Заголовок: {article['title']}")
    print(f"Дата: {article['published']}")
    print(f"Ссылка: {article['link']}")
    print(f"Описание: {article['summary'][:100]}...")

Yahoo Finance:

Статья 1
Заголовок: The Smartest Technology Stock to Buy With $200 Right Now
Дата: 2025-11-01T11:30:00Z
Ссылка: https://finance.yahoo.com/news/smartest-technology-stock-buy-200-113000723.html
Описание: No summary...

Статья 2
Заголовок: Ray Dalio says choosing the right partner is life’s most important decision — make sure you align on these 3 money musts
Дата: 2025-11-01T11:30:00Z
Ссылка: https://finance.yahoo.com/news/ray-dalio-says-choosing-partner-113000207.html
Описание: No summary...

Статья 3
Заголовок: A tax-refund surge is coming, JPMorgan strategist says — and it’ll shift US economy like a new round of stimulus checks
Дата: 2025-11-01T11:37:00Z
Ссылка: https://finance.yahoo.com/news/tax-refund-surge-coming-jpmorgan-113700564.html
Описание: No summary...

Статья 4
Заголовок: S&P 500 Giants JPMorgan, Eli Lilly Lead Five Stocks Near Buy Points
Дата: 2025-11-01T12:00:23Z
Ссылка: https://www.investors.com/news/sp-500-jpmorgan-eli-lilly-stocks-buy-points/?src=A00220

In [9]:
#датасет с новостями из Kaggle
#import kagglehub
#path = kagglehub.dataset_download("suruchiarora/top-25-world-news-2018-2023")
#print("Path to dataset files:", path)


df = pd.read_csv("WorldNewsData.csv")
print(df.head(5))

Path to dataset files: C:\Users\denis\.cache\kagglehub\datasets\suruchiarora\top-25-world-news-2018-2023\versions\1

           Date                                               Top1                                               Top2                                               Top3                                               Top4                                               Top5                                               Top6                                               Top7                                               Top8                                               Top9                                              Top10                                              Top11                                              Top12                                              Top13                                              Top14                                              Top15                                              Top16                                              Top17 